# LangChain Smart Prompt Orchestrator

## Goal

Learn how modern LLM applications are built using LangChain.

This project will progress through several phases:

1. Foundations
2. Prompt Templates
3. Chains
4. Routing
5. Project Structure
6. FastAPI
7. Streamlit / Gradio
8. Docker and Deployment

---


```
## Final Architecture

User Request
      ↓
Router
      ↓
Choose Chain
      ↓
PromptTemplate
      ↓
LLM
      ↓
Response
```
---

This notebook begins with the foundations required to understand the architecture before building it.

# What is this notebook?

This notebook is not intended to build the final application.

Its purpose is to understand the fundamental concepts that appear in LangChain applications.

By the end of this notebook, I should understand:

- What an LLM is
- What a Prompt is
- What a PromptTemplate is
- What a Chain is
- What a Router is

These concepts will later be converted into Python modules and APIs.

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [1]:
# ==========================================
# CELL 3 — ENVIRONMENT CHECK
# ------------------------------------------
# Purpose:
# Verify that the notebook is running inside
# the correct Python environment.
# ==========================================

import sys

print("Python executable:")
print(sys.executable)

Python executable:
c:\Users\User\anaconda3\envs\llm-env\python.exe


# Foundation Concept 1 — What is an LLM?

LLM stands for Large Language Model.

A Large Language Model is a system trained on large amounts of text that can generate human-like responses.

Examples:

- GPT models
- Llama models
- Mistral models
- Gemma models

An LLM receives text as input and produces text as output.

Basic flow:

```text
User Input
      ↓
LLM
      ↓
Generated Response

# Foundation Concept 2 — What is a Prompt?

A prompt is the instruction given to an LLM.

Examples:

Prompt:
"Explain machine learning."

Prompt:
"Summarize this article."

Prompt:
"Generate three startup ideas."

The quality of the response often depends on the quality of the prompt.

Basic flow:

```text
Prompt
   ↓
LLM
   ↓
Response
```

Examples:

Prompt:
"What is machine learning?"

Response:
"Machine learning is a field of artificial intelligence that enables systems to learn patterns from data."

Prompt:
"Give me three startup ideas related to fitness."

Response:
"A fitness coaching app, a nutrition tracking platform, and an AI workout planner."

In LangChain, prompts become reusable objects called PromptTemplates.

# Foundation Concept 3 — What is a Chain?

A chain is a sequence of steps executed in a specific order.

Without a chain:

User
  ↓
LLM
  ↓
Response

With a chain:

User
  ↓
Prompt
  ↓
LLM
  ↓
Response

A chain allows us to organize the workflow.

Example:

User asks:
"Explain machine learning"

The chain may:

1. Build a prompt
2. Send the prompt to the LLM
3. Receive the response
4. Return the response to the user

Visual representation:

```text
Input
  ↓
Prompt
  ↓
LLM
  ↓
Output
```

In LangChain, chains connect components together.

A chain does not need to be complex.

Even:

Prompt → LLM

is already a chain.

# Foundation Concept 4 — What is a Router?

A router is a component that decides which workflow should be executed.

Instead of sending every request to the same chain, the router analyzes the request and chooses the most appropriate chain.

Example:

User:
"Explain machine learning"

Router:
→ explain_chain

User:
"Summarize this article"

Router:
→ summarize_chain

User:
"Generate startup ideas"

Router:
→ idea_chain

Visual representation:

```text
                 ┌───────────────┐
                 │ User Request  │
                 └───────┬───────┘
                         ↓
                  ┌───────────┐
                  │  Router   │
                  └─────┬─────┘
                        │
          ┌─────────────┼─────────────┐
          ↓             ↓             ↓
   explain_chain summarize_chain idea_chain
```

The router is responsible for deciding which chain should handle the request.

In this project, we will first build a simple router using Python rules.

Later, we will connect it to real LangChain chains.

In [2]:
# ==========================================
# CELL 8 — SIMPLE PYTHON ROUTER
# ------------------------------------------
# Purpose:
# Simulate how a router decides which
# chain should handle a user request.
#
# No LangChain yet.
# Just Python.
# ==========================================

def route_request(user_request):
    
    user_request = user_request.lower()

    if "explain" in user_request:
        return "explain_chain"

    elif "summarize" in user_request:
        return "summarize_chain"

    elif "idea" in user_request:
        return "idea_chain"

    else:
        return "default_chain"


request = "Explain machine learning"

selected_chain = route_request(request)

print("User Request:")
print(request)

print("\nSelected Chain:")
print(selected_chain)

User Request:
Explain machine learning

Selected Chain:
explain_chain


In [3]:
# ==========================================
# CELL 9 — MULTIPLE REQUESTS
# ------------------------------------------
# Purpose:
# Show how a router handles different
# types of requests.
# ==========================================

def route_request(user_request):

    user_request = user_request.lower()

    if "explain" in user_request:
        return "explain_chain"

    elif "summarize" in user_request:
        return "summarize_chain"

    elif "idea" in user_request:
        return "idea_chain"

    else:
        return "default_chain"


requests = [
    "Explain machine learning",
    "Summarize this article",
    "Give me startup ideas",
    "Tell me a joke"
]

for request in requests:

    selected_chain = route_request(request)

    print(f"Request: {request}")
    print(f"Chain: {selected_chain}")
    print("-" * 40)

Request: Explain machine learning
Chain: explain_chain
----------------------------------------
Request: Summarize this article
Chain: summarize_chain
----------------------------------------
Request: Give me startup ideas
Chain: idea_chain
----------------------------------------
Request: Tell me a joke
Chain: default_chain
----------------------------------------


In [4]:
# ==========================================
# CELL 10 — FAKE CHAINS
# ------------------------------------------
# Purpose:
# Create simple chains using Python
# functions.
#
# No LangChain yet.
# ==========================================

def explain_chain(topic):
    
    return f"Explanation about: {topic}"


def summarize_chain(text):
    
    return f"Summary of: {text}"


def idea_chain(topic):
    
    return f"Three startup ideas about: {topic}"


print(explain_chain("machine learning"))
print(summarize_chain("Artificial intelligence is transforming industries"))
print(idea_chain("fitness"))

Explanation about: machine learning
Summary of: Artificial intelligence is transforming industries
Three startup ideas about: fitness


In [5]:
# ==========================================
# CELL 11 — ROUTER EXECUTES CHAINS
# ------------------------------------------
# Purpose:
# The router now selects AND executes
# the appropriate chain.
# ==========================================

def explain_chain(topic):

    return f"Explanation about: {topic}"


def summarize_chain(text):

    return f"Summary of: {text}"


def idea_chain(topic):

    return f"Three startup ideas about: {topic}"


def route_request(user_request):

    request = user_request.lower()

    if "explain" in request:

        return explain_chain("machine learning")

    elif "summarize" in request:

        return summarize_chain("Artificial intelligence article")

    elif "idea" in request:

        return idea_chain("fitness")

    else:

        return "No chain available"


response = route_request(
    "Explain machine learning"
)

print(response)

Explanation about: machine learning


In [6]:
# ==========================================
# CELL 12 — DYNAMIC ROUTER
# ------------------------------------------
# Purpose:
# Use the user's request as input
# instead of hardcoded values.
# ==========================================

def explain_chain(user_request):

    return f"Explanation generated for: {user_request}"


def summarize_chain(user_request):

    return f"Summary generated for: {user_request}"


def idea_chain(user_request):

    return f"Startup ideas generated for: {user_request}"


def route_request(user_request):

    request = user_request.lower()

    if "explain" in request:

        return explain_chain(user_request)

    elif "summarize" in request:

        return summarize_chain(user_request)

    elif "idea" in request:

        return idea_chain(user_request)

    else:

        return "No chain available"


print(
    route_request(
        "Explain machine learning"
    )
)

print()

print(
    route_request(
        "Summarize this article"
    )
)

print()

print(
    route_request(
        "Give me startup ideas"
    )
)

Explanation generated for: Explain machine learning

Summary generated for: Summarize this article

Startup ideas generated for: Give me startup ideas


In [7]:
# ==========================================
# CELL 13 — ROUTER WITH CHAIN REGISTRY
# ------------------------------------------
# Purpose:
# Store chains in a dictionary and let
# the router select them dynamically.
# ==========================================

def explain_chain(user_request):

    return f"Explanation generated for: {user_request}"


def summarize_chain(user_request):

    return f"Summary generated for: {user_request}"


def idea_chain(user_request):

    return f"Startup ideas generated for: {user_request}"


CHAINS = {
    "explain": explain_chain,
    "summarize": summarize_chain,
    "idea": idea_chain
}


def route_request(user_request):

    request = user_request.lower()

    for keyword, chain in CHAINS.items():

        if keyword in request:

            return chain(user_request)

    return "No chain available"


requests = [
    "Explain machine learning",
    "Summarize this article",
    "Give me startup ideas",
    "Tell me a joke"
]

for request in requests:

    response = route_request(request)

    print(f"Request: {request}")
    print(f"Response: {response}")
    print("-" * 50)

Request: Explain machine learning
Response: Explanation generated for: Explain machine learning
--------------------------------------------------
Request: Summarize this article
Response: Summary generated for: Summarize this article
--------------------------------------------------
Request: Give me startup ideas
Response: Startup ideas generated for: Give me startup ideas
--------------------------------------------------
Request: Tell me a joke
Response: No chain available
--------------------------------------------------


In [1]:
# ==========================================
# CELL 14 — PROMPT TEMPLATE
# ------------------------------------------
# Purpose:
# Import LangChain's PromptTemplate
# ==========================================

from langchain_core.prompts import PromptTemplate

print("PromptTemplate imported successfully.")

c:\Users\User\anaconda3\envs\llm-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PromptTemplate imported successfully.


This cell defines a LangChain PromptTemplate, which is a structured way to build prompts dynamically instead of writing fixed text. The role of LangChain here is to separate prompt design from data input, so you can reuse the same template with different topics.

The {topic} part is a placeholder that will be filled later. When you call .format(topic="machine learning"), LangChain simply replaces that placeholder and produces a fully written prompt string.

In this stage there is no LLM or chain yet involved—this cell only prepares the input structure that will later be used inside LangChain chains to feed the model in a consistent way.

In [3]:
from langchain_core.prompts import PromptTemplate

explain_prompt = PromptTemplate.from_template(
    """
    You are a helpful teacher.

    Explain the following topic simply:

    Topic: {topic}
    """
)

print(
    explain_prompt.format(
        topic="machine learning"
    )
)


    You are a helpful teacher.

    Explain the following topic simply:

    Topic: machine learning
    


This cell introduces LangChain’s PromptTemplate system, which is the first building block of a LangChain workflow. The main idea is to stop writing fixed prompts manually and instead define reusable templates that can dynamically receive inputs. Each template here represents a different task: explanation, summarization, and idea generation.

From a technical perspective, LangChain is being used to create structured prompt objects instead of raw strings. The PromptTemplate.from_template() function allows you to define placeholders like {topic} or {text}, which are later filled using .format(). This makes prompts more flexible, reusable, and less error-prone when scaling to multiple tasks or pipelines.

Conceptually, this is the foundation of LangChain’s philosophy: separating prompt design from execution. Right now, you are only defining “instructions for the model,” but not yet running an LLM or a chain. Later, these templates will be connected to models inside LCEL chains, where LangChain will manage the full flow: input → prompt → LLM → output.

In [5]:
# ==========================================
# CELL 15 — MULTIPLE PROMPT TEMPLATES
# ------------------------------------------
# Purpose:
# Create reusable prompts for different
# tasks.
# ==========================================

from langchain_core.prompts import PromptTemplate


explain_prompt = PromptTemplate.from_template(
    """
    You are a helpful teacher.

    Explain the following topic simply:

    Topic: {topic}
    """
)


summarize_prompt = PromptTemplate.from_template(
    """
    Summarize the following text:

    {text}
    """
)


idea_prompt = PromptTemplate.from_template(
    """
    Generate 3 startup ideas about:

    {topic}
    """
)


print("Explain Prompt")
print(
    explain_prompt.format(
        topic="machine learning"
    )
)

print("\n" + "=" * 50 + "\n")

print("Summarize Prompt")
print(
    summarize_prompt.format(
        text="Artificial intelligence is transforming industries."
    )
)

print("\n" + "=" * 50 + "\n")

print("Idea Prompt")
print(
    idea_prompt.format(
        topic="fitness"
    )
)

Explain Prompt

    You are a helpful teacher.

    Explain the following topic simply:

    Topic: machine learning
    


Summarize Prompt

    Summarize the following text:

    Artificial intelligence is transforming industries.
    


Idea Prompt

    Generate 3 startup ideas about:

    fitness
    


-----------------------------------------------
Cell 16 — Conceptual explanation (LangChain + “chains” idea)

This cell is your first “chain abstraction step”, even though it still uses plain Python functions under the hood. The key idea is that you are starting to separate responsibilities: prompts are no longer just strings you print, they become reusable templates, and functions act like simple “pipelines” that take an input and return a formatted prompt.

1. Role of LangChain here

You are using langchain_core.prompts.PromptTemplate to define structured prompt builders. Instead of manually writing prompts every time, LangChain lets you define templates with placeholders like {topic} or {text}. This is important because in real LLM systems, consistency and reuse of prompts is critical.

So LangChain here is not “running the model yet” — it is helping you build standardized prompt components that can later be plugged into chains, routers, or full pipelines.

2. What a “chain” means in this cell

Even though you define functions like explain_chain(topic), summarize_chain(text), and idea_chain(topic), these are not real LangChain LCEL chains yet.

Right now they are:

Input → formatted prompt string → output string

So conceptually, this is a manual pipeline, but it prepares you for LCEL where you will replace these functions with real composable components like:

PromptTemplate → LLM → OutputParser
3. Why this step matters in the learning path

This cell is teaching you the transition point:

Before: static prompts (just strings)
Now: reusable prompt templates (LangChain abstraction)
Next: real LCEL chains (composable objects)
Later: routing + multi-chain systems + modular architecture

So this is basically the bridge between “Python scripting” and “LangChain architecture thinking”.

In [4]:
# ==========================================
# CELL 16 — LANGCHAIN CHAINS
# ------------------------------------------
# Purpose:
# Replace fake chain responses with
# PromptTemplate-generated prompts.
# ==========================================

from langchain_core.prompts import PromptTemplate


explain_prompt = PromptTemplate.from_template(
    """
    You are a helpful teacher.

    Explain the following topic simply:

    Topic: {topic}
    """
)


summarize_prompt = PromptTemplate.from_template(
    """
    Summarize the following text:

    {text}
    """
)


idea_prompt = PromptTemplate.from_template(
    """
    Generate 3 startup ideas about:

    {topic}
    """
)


def explain_chain(topic):

    return explain_prompt.format(
        topic=topic
    )


def summarize_chain(text):

    return summarize_prompt.format(
        text=text
    )


def idea_chain(topic):

    return idea_prompt.format(
        topic=topic
    )


print(explain_chain("machine learning"))

print("\n" + "=" * 50 + "\n")

print(summarize_chain(
    "Artificial intelligence is transforming industries."
))

print("\n" + "=" * 50 + "\n")

print(idea_chain("fitness"))


    You are a helpful teacher.

    Explain the following topic simply:

    Topic: machine learning
    



    Summarize the following text:

    Artificial intelligence is transforming industries.
    



    Generate 3 startup ideas about:

    fitness
    


In [7]:
import warnings
warnings.filterwarnings("ignore")

------------------------------------------------------------------------
This cell introduces the idea of prompt abstraction using LangChain’s PromptTemplate, which is one of the core building blocks in LangChain. Instead of writing raw strings every time you want to ask a model something, you define structured templates with placeholders like {topic} or {text}. These templates act as reusable “question formats” that can be dynamically filled later. This is the first step toward building systems that are modular, predictable, and easier to scale than manually writing prompts every time.

From a LangChain perspective, the key idea is separation of concerns. The PromptTemplate object is responsible only for defining how the input should be formatted, not for executing any model or logic. LangChain provides this abstraction so that prompts become first-class objects in a pipeline. The .from_template() method converts a simple string into a structured template that LangChain can later integrate into a full chain (Prompt → LLM → Output Parser). Even in this simple cell, you're already preparing the foundation for LCEL-style pipelines.

Finally, when you call .format(), you're manually testing how the prompt will look once variables are injected. This is important because it lets you validate prompt structure before connecting it to an LLM. In real LangChain chains, you typically don’t stop at .format()—instead, the template is passed into a pipeline where the LLM consumes it directly. So this cell is essentially a design and validation step for prompt engineering, preparing you for the next evolution: chaining prompts with models and output parsers in a fully automated workflow.

In [5]:
# ==========================================
# CELL 17 — LOCAL LLM SETUP
# ------------------------------------------
# Purpose:
# Load a small free Hugging Face model
# ==========================================

from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    max_new_tokens=128
)

print("LLM loaded successfully.")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4417.26it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM'

LLM loaded successfully.


-----------------------------------------------------------------
This cell combines LangChain prompt engineering with raw Hugging Face model inference, showing the full manual pipeline before using LCEL or higher-level abstractions. First, you load the tokenizer and model directly using AutoTokenizer and AutoModelForSeq2SeqLM. This is Hugging Face’s low-level interface, where you explicitly control how text is converted into tokens and how the model generates output. At this stage, LangChain is only used for prompt creation, not execution.

Then, you use PromptTemplate from LangChain to define a structured instruction for the model. The template ensures that the input follows a consistent format, and .format() is used to inject the actual topic into the prompt. This step shows how LangChain helps standardize input construction, even when the model execution is handled outside of LangChain.

Finally, you manually connect everything: the formatted prompt is tokenized, passed into the model using model.generate(), and decoded back into text. This simulates what a full LangChain chain would eventually automate. Here you are explicitly controlling the entire flow — Prompt → Tokenizer → Model → Output — which is useful for understanding how LCEL and LangChain chains abstract these steps later.

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.prompts import PromptTemplate

# -----------------------------
# 1. Load model (NO pipeline)
# -----------------------------
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# -----------------------------
# 2. PromptTemplate
# -----------------------------
explain_prompt = PromptTemplate.from_template(
    """
    You are a helpful teacher.

    Explain the following topic simply:

    Topic: {topic}
    """
)

# -----------------------------
# 3. Create prompt
# -----------------------------
prompt = explain_prompt.format(topic="machine learning")

print("=== PROMPT ===")
print(prompt)

# -----------------------------
# 4. DIRECT GENERATION (NO PIPELINE)
# -----------------------------
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=128)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n=== RESPONSE ===")
print(result)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2969.02it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


=== PROMPT ===

    You are a helpful teacher.

    Explain the following topic simply:

    Topic: machine learning
    

=== RESPONSE ===
Machine learning is a learning tool.


In [8]:
prompt = explain_prompt.format(topic="machine learning")

response = llm(prompt)

print(response)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '\n    You are a helpful teacher.\n\n    Explain the following topic simply:\n\n    Topic: machine learning\n    '}]


---------------------------------------------------------------------------------------
This cell builds the **first complete AI orchestration system**, combining Hugging Face (model execution), LangChain (prompt structuring), and manual routing logic. Conceptually, it is simulating what LangChain LCEL and agents will later automate: taking a user input, deciding what to do, formatting a prompt, and generating a response.

The `run_llm()` function is your **manual inference engine**, where you directly control the full Hugging Face flow: text → tokenization → model generation → decoding. This is the “raw brain execution layer” of your system. Above it, you define `PromptTemplate`s using LangChain, which act as structured instruction builders. These templates ensure that each task (explain, summarize, idea generation) has a consistent format before being sent to the model.

Finally, the `router()` function represents a **basic decision system (rule-based agent)**. It checks keywords in the user input and selects which “chain” (function) to execute. Each chain combines LangChain prompts with the LLM function, forming a mini pipeline: Prompt → LLM → Output. This cell is important because it manually reproduces what LangChain LCEL will later abstract: routing logic, chain composition, and execution flow.


In [7]:
# ==========================================
# CELL 19 — SMART ORCHESTRATOR (FIRST VERSION)
# ------------------------------------------
# Goal:
# Build a simple AI system that mimics a
# LangChain-style architecture using:
#
# User Input
#     ↓
# Router (decides what to do)
#     ↓
# PromptTemplate (builds prompt)
#     ↓
# LLM (generates answer)
#
# Key idea:
# We are manually implementing what LangChain
# automates later.
# ==========================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.prompts import PromptTemplate

# ==========================================
# 1. LOAD MODEL (shared across all chains)
# ==========================================

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


def run_llm(prompt_text):
    """
    Converts prompt → tokens → model output → readable text
    """
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ==========================================
# 2. PROMPT TEMPLATES
# ==========================================

explain_prompt = PromptTemplate.from_template(
    "Explain this topic simply: {topic}"
)

summarize_prompt = PromptTemplate.from_template(
    "Summarize this text: {text}"
)

idea_prompt = PromptTemplate.from_template(
    "Give 3 startup ideas about: {topic}"
)


# ==========================================
# 3. CHAINS (Prompt → LLM)
# ==========================================

def explain_chain(user_input):
    prompt = explain_prompt.format(topic=user_input)
    return run_llm(prompt)


def summarize_chain(user_input):
    prompt = summarize_prompt.format(text=user_input)
    return run_llm(prompt)


def idea_chain(user_input):
    prompt = idea_prompt.format(topic=user_input)
    return run_llm(prompt)


# ==========================================
# 4. ROUTER (decides which chain runs)
# ==========================================

def router(user_request):
    req = user_request.lower()

    if "explain" in req:
        return explain_chain(user_request)

    elif "summarize" in req:
        return summarize_chain(user_request)

    elif "idea" in req:
        return idea_chain(user_request)

    else:
        return "No valid chain found"


# ==========================================
# 5. TEST CASES
# ==========================================

tests = [
    "Explain machine learning",
    "Summarize AI is transforming industries",
    "Give me startup ideas about fitness"
]

for t in tests:
    print("\n====================")
    print("INPUT:", t)
    print("OUTPUT:", router(t))

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3392.63it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



INPUT: Explain machine learning
OUTPUT: Machine learning is a process that requires a machine to learn.

INPUT: Summarize AI is transforming industries
OUTPUT: AI is transforming industries

INPUT: Give me startup ideas about fitness
OUTPUT: a gym


- The router is like a traffic controller. It looks at what the user is asking and decides what type of task it is. For example, if the user says “explain machine learning,” it chooses the explanation path. If they say “summarize this text,” it chooses the summarization path. It doesn’t generate content — it only decides the direction.

- The PromptTemplate is like a structured question builder. It takes the user’s input and puts it into a clear format that the model can understand better. Instead of sending random text, it turns it into a well-designed instruction like “Explain this topic simply: machine learning.” So it defines how we ask the question.

- The LLM (language model) is the part that actually answers. It reads the formatted prompt and generates the response based on what it has learned from data. It doesn’t decide what to do or how to ask — it only focuses on producing the best possible answer.

-----------------------------------------------------------------------------------------------
This cell represents a clean and stabilized version of your pipeline, where you explicitly separate two responsibilities: prompt creation and model execution. Instead of mixing everything into a single function or LCEL chain, you first use LangChain’s PromptTemplate to build a structured instruction, and then you pass the resulting string into a safe Hugging Face inference function.

The run_llm() function is a manual execution layer that directly handles tokenization, model inference, and decoding using transformers. This is still outside LCEL, but it is intentionally isolated to avoid earlier errors where LangChain components were incorrectly passing non-string objects. The key design idea here is that the model only receives a clean string input, ensuring stability and preventing type mismatches.

From a system design perspective, this cell shows a hybrid architecture: LangChain is used for prompt standardization, while Hugging Face is used for controlled inference. Even though LCEL is mentioned, this is not full LCEL yet—it is a safer intermediate step where you manually enforce the flow: PromptTemplate → formatted string → LLM → output.

In [2]:
# ==========================================
# CELL 20 — LCEL (STABLE VERSION)
# ------------------------------------------
# FIX:
# We separate:
#   1. Prompt formatting (LCEL)
#   2. Model execution (manual safe function)
# ==========================================

from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==========================================
# MODEL
# ==========================================

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ==========================================
# SAFE LLM FUNCTION (expects STRING only)
# ==========================================

def run_llm(prompt_text: str) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ==========================================
# PROMPT TEMPLATE
# ==========================================

explain_prompt = PromptTemplate.from_template(
    """
You are a simple and clear teacher.

Write EXACTLY:
- 2 short sentences
- 1 real-world example

Topic: {topic}
"""
)

# ==========================================
# STEP 1 — format prompt FIRST
# ==========================================

prompt = explain_prompt.format(topic="machine learning")

# ==========================================
# STEP 2 — send clean string to model
# ==========================================

result = run_llm(prompt)

print("=== OUTPUT ===")
print(result)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3280.14it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


=== OUTPUT ===
Machine learning is a very important part of the human brain.


-----------------------------------------------------------------------------------------------
LCEL = LangChain Expression Language

It is LangChain’s way of building AI workflows using composable pipelines, where each step is connected with | instead of manual function calls.

🧠 What this cell does (conceptually)

This cell builds a first LCEL routing pipeline, where an input dictionary flows through three stages:

Input → Router → Prompt Builder → LLM → Output

The goal is to simulate a real AI system architecture where:

one component decides what to do (router)
another builds the prompt
another runs the model

Even though the router is simple here, it introduces the idea of multi-step orchestration using LCEL.

🔧 What the code is doing
router() → receives a dictionary and returns structured data (currently just passes "topic" through)
RunnableLambda(router) → converts the Python function into an LCEL-compatible component
build_prompt() → transforms structured input into a final prompt string
RunnableLambda(run_llm) → wraps your Hugging Face inference function so it can be used inside LCEL
full_chain = route_chain | prompt_chain | RunnableLambda(run_llm) → creates the full pipeline

So the execution becomes automatic:

input dict → router → prompt string → model → final answer

⚙️ Why this is important

This is your first real LCEL structure because you are no longer calling functions manually. Instead, LangChain is handling the flow of data between components automatically using the | operator.

It is the foundation for:

multi-chain systems
advanced routing (different behaviors per input)
scalable LLM pipelines

In short: this cell transforms your previous manual logic into a modular AI pipeline architecture using LCEL.

In [8]:
# ==========================================
# CELL 21 — LCEL ROUTER (FIXED VERSION)
# ------------------------------------------
# FIX:
# - NO nested .invoke()
# - NO hardcoded input
# - Proper LCEL flow
# ==========================================

from langchain_core.runnables import RunnableLambda

# ==========================================
# 1. ROUTER (simple rule-based)
# ==========================================

def router(input_dict):
    topic = input_dict["topic"].lower()

    return {
        "topic": input_dict["topic"]
    }

route_chain = RunnableLambda(router)

# ==========================================
# 2. SAFE LLM FUNCTION (unchanged)
# ==========================================

def run_llm(prompt_text: str) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ==========================================
# 3. PROMPT PIPELINE (explicit step)
# ==========================================

def build_prompt(x):
    return f"Explain this topic simply in 2 sentences with an example: {x['topic']}"

prompt_chain = RunnableLambda(build_prompt)

# ==========================================
# 4. FULL PIPELINE (CLEAN LCEL STYLE)
# ==========================================

full_chain = route_chain | prompt_chain | RunnableLambda(run_llm)

# ==========================================
# 5. TEST
# ==========================================

result = full_chain.invoke({"topic": "machine learning"})

print("=== OUTPUT ===")
print(result)

=== OUTPUT ===
machine learning


==============================================================================================================================
This cell builds a complete LCEL routing system using RunnableBranch, which is LangChain’s way of selecting different execution paths based on conditions. The goal here is to simulate a real intelligent pipeline where the system decides which chain to execute depending on the input. In this case, the system only has two paths: an “explain” chain and a fallback response.

The core of the system is the safe LLM function (run_llm), which ensures that whatever LCEL sends (string, dict, or object) is converted into a clean string before being passed to the Hugging Face tokenizer. This is critical because earlier errors came from mismatched input types between LCEL components and the model. Then, the explain_chain combines a PromptTemplate, the LLM function, and an output parser into a single LCEL pipeline.

Finally, RunnableBranch acts as the router logic of the system, using the is_explain function to decide whether to run the explanation chain or the fallback chain. The final execution router_chain.invoke(...) shows a full LCEL workflow: input → condition check → chain selection → prompt generation → model execution → output. This is your first stable version of a multi-chain LCEL routing architecture, where logic, prompts, and model execution are fully modular and connected.

-------------------------------------------------------------------------------------------------------------------------------------
1. Prompt Template

This defines how the model should be asked a question. Instead of writing raw text every time, you create a structured format (Explain this topic... {topic}). Conceptually, this is your instruction layer, where you control the behavior of the LLM without touching the model itself.

2. Safe LLM Function

This is your execution engine wrapper around the Hugging Face model. It takes a prompt, converts it into tokens, runs the model, and returns text. Conceptually, this is the “brain execution layer”, but protected so it always receives clean string input to avoid crashes.

3. Explain Chain

This combines:

PromptTemplate → LLM → OutputParser

Conceptually, this is your first real LCEL pipeline, where you connect components into a single flow. It represents a reusable “skill”: in this case, the ability to explain topics.

4. Routing Condition (is_explain)

This is a decision function that analyzes user input and decides whether the system should use the explanation chain. Conceptually, it is your rule-based classifier, a very simple form of “intelligence” that selects behavior based on keywords.

5. Fallback Chain

This is the default response path when no condition matches. Conceptually, it acts as a safety net or guardrail, ensuring the system always returns something instead of failing or crashing when input is unexpected.

6. LCEL Router (RunnableBranch)

This is the core decision engine of the system. It takes the routing condition and maps it to different chains. Conceptually, this is your control center, where the system chooses which “AI behavior” to execute based on input.

7. Test Execution

This is where you actually run the full system. Conceptually, it represents the end-to-end pipeline execution, showing how raw input flows through routing → selection → prompt creation → LLM → final output. This is the moment where all components become a real working AI system.


In [9]:
# ==========================================
# CELL 23 — LCEL ROUTER (FULL STABLE VERSION)
# ------------------------------------------
# Goal:
# - RunnableBranch routing
# - SAFE input handling for HuggingFace model
# - No type errors between LCEL and tokenizer
# ==========================================

from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

# ==========================================
# 1. PROMPT TEMPLATE
# ==========================================

explain_prompt = PromptTemplate.from_template(
    "Explain this topic simply in 2 sentences with an example:\n{topic}"
)

# ==========================================
# 2. SAFE LLM FUNCTION (CRITICAL FIX)
# ==========================================
# This prevents crashes when LCEL sends dicts or objects

def run_llm(prompt_text):
    # Normalize input to string (VERY IMPORTANT)
    if isinstance(prompt_text, dict):
        prompt_text = prompt_text.get("topic", str(prompt_text))

    prompt_text = str(prompt_text)

    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ==========================================
# 3. EXPLAIN CHAIN
# ==========================================

explain_chain = (
    explain_prompt
    | RunnableLambda(run_llm)
    | StrOutputParser()
)

# ==========================================
# 4. ROUTING CONDITION
# ==========================================
# Simple rule-based classifier (we will upgrade later)

def is_explain(input_dict):
    topic = input_dict.get("topic", "").lower()

    return (
        "explain" in topic or
        "what" in topic or
        "machine" in topic
    )

# ==========================================
# 5. FALLBACK CHAIN
# ==========================================

fallback_chain = RunnableLambda(
    lambda x: "I can only explain topics for now."
)

# ==========================================
# 6. LCEL ROUTER (RunnableBranch)
# ==========================================

router_chain = RunnableBranch(
    (is_explain, explain_chain),
    fallback_chain
)

# ==========================================
# 7. TEST EXECUTION
# ==========================================

result = router_chain.invoke(
    {"topic": "machine learning"}
)

print("=== LCEL ROUTED OUTPUT ===")
print(result)

=== LCEL ROUTED OUTPUT ===
machine learning


In [10]:
# ==========================================
# CELL 24 — LLM ROUTER (INTELLIGENT ROUTING)
# ==========================================
# PURPOSE:
# Build a routing system where an LLM decides
# which chain to execute.
#
# ARCHITECTURE:
# User Input
#   ↓
# Router LLM (decides: explain or other)
#   ↓
# RunnableBranch (routes execution)
#   ↓
# Final Chain Output
# ==========================================

from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 1. ROUTER PROMPT (LLM DECISION MAKER)
# ==========================================
# This prompt forces the model to classify the request

router_prompt = PromptTemplate.from_template(
    """
You are a routing system.

Your job is to decide what to do with the user input.

Return ONLY ONE word:
- explain
- other

User input:
{topic}
"""
)

# ==========================================
# 2. SAFE LLM FUNCTION
# ==========================================
# Ensures compatibility between LangChain and HF model

def run_llm(prompt_text):
    # Normalize input type (VERY IMPORTANT)
    if isinstance(prompt_text, dict):
        prompt_text = prompt_text.get("topic", str(prompt_text))

    prompt_text = str(prompt_text)

    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ==========================================
# 3. ROUTER CHAIN (LLM DECISION)
# ==========================================
# The LLM decides the route (not rules anymore)

router_chain = (
    router_prompt
    | RunnableLambda(run_llm)
    | StrOutputParser()
)

# ==========================================
# 4. EXPLAIN CHAIN
# ==========================================
# This chain is executed when route = "explain"

explain_prompt = PromptTemplate.from_template(
    "Explain this topic simply in 2 sentences with an example:\n{topic}"
)

explain_chain = (
    explain_prompt
    | RunnableLambda(run_llm)
    | StrOutputParser()
)

# ==========================================
# 5. ROUTE DECISION FUNCTION
# ==========================================
# Converts LLM output into a clean routing label

def route_selector(route_output):
    route_output = route_output.strip().lower()

    if "explain" in route_output:
        return "explain"
    else:
        return "other"

# ==========================================
# 6. FALLBACK CHAIN
# ==========================================

fallback_chain = RunnableLambda(
    lambda x: "I can only explain topics for now."
)

# ==========================================
# 7. FULL LCEL PIPELINE
# ==========================================
# Flow:
# Input → Router LLM → Branch → Chain → Output

full_chain = (
    router_chain
    | RunnableLambda(lambda route: {
        "route": route_selector(route),
        "topic": "machine learning"
    })
    | RunnableBranch(
        (lambda x: x["route"] == "explain", explain_chain),
        fallback_chain
    )
)

# ==========================================
# 8. TEST EXECUTION
# ==========================================

result = full_chain.invoke({"topic": "machine learning"})

print("=== LLM ROUTER OUTPUT ===")
print(result)

=== LLM ROUTER OUTPUT ===
I can only explain topics for now.


In [9]:
# ==========================================================
# CELL 25 — FIX LCEL + TOKENIZER CONTRACT (STABLE VERSION)
# ----------------------------------------------------------
# PURPOSE:
# Fix the mismatch between LangChain dict inputs and
# HuggingFace tokenizer string requirements.
#
# RULE:
# LCEL passes dict → we must extract "topic" safely.
# ==========================================================

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.output_parsers import StrOutputParser

# ----------------------------------------------------------
# 1. ROUTER FUNCTION
# ----------------------------------------------------------
def route_input(inputs: dict) -> str:
    text = inputs["topic"].lower()

    if "example" in text:
        return "example"
    elif "what" in text or "explain" in text:
        return "explain"
    else:
        return "fallback"


# ----------------------------------------------------------
# 2. PROMPTS
# ----------------------------------------------------------
explain_prompt = PromptTemplate.from_template(
    "Explain this topic simply:\n{topic}"
)

example_prompt = PromptTemplate.from_template(
    "Give a simple real-world example of:\n{topic}"
)

fallback_prompt = PromptTemplate.from_template(
    "Answer briefly:\n{topic}"
)


# ----------------------------------------------------------
# 3. SAFE LLM WRAPPER (IMPORTANT FIX)
# ----------------------------------------------------------
def run_llm(inputs: dict) -> str:
    """
    FIX:
    LCEL passes dict → we extract topic safely → convert to string
    """
    if isinstance(inputs, dict):
        text = inputs.get("topic", "")
    else:
        text = str(inputs)

    inputs_tok = tokenizer(text, return_tensors="pt")
    outputs = model.generate(**inputs_tok, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ----------------------------------------------------------
# 4. CHAINS
# ----------------------------------------------------------
explain_chain = explain_prompt | RunnableLambda(run_llm) | StrOutputParser()
example_chain = example_prompt | RunnableLambda(run_llm) | StrOutputParser()
fallback_chain = fallback_prompt | RunnableLambda(run_llm) | StrOutputParser()


# ----------------------------------------------------------
# 5. ROUTER + BRANCHING
# ----------------------------------------------------------
full_chain = RunnableBranch(
    (lambda x: route_input(x) == "explain", explain_chain),
    (lambda x: route_input(x) == "example", example_chain),
    fallback_chain
)


# ----------------------------------------------------------
# 6. TEST
# ----------------------------------------------------------
result = full_chain.invoke({"topic": "machine learning example"})

print("=== LCEL ROUTED OUTPUT ===")
print(result)

=== LCEL ROUTED OUTPUT ===
a machine learning example


----------------------------------------------------------------------------------------------------------------------
This cell is not changing your model or your routing system. Instead, it is improving the quality of the answers by rewriting the prompts.

Think of it like this:

The model is the same brain, but now you are giving it better instructions, so it behaves better.

So this cell is about prompt engineering improvements, not architecture changes.

🔧 Part 1 — Explain Prompt

This prompt tells the model how to explain a topic.

In simple terms:

It forces the model to act like a beginner-friendly teacher
It limits the answer to 2–3 short sentences
It encourages simple language and analogies

👉 Purpose:
Make explanations clear, short, and easy to understand, instead of vague or long answers.

🔧 Part 2 — Example Prompt

This prompt changes the behavior from “explain” to “give a real example”.

In simple terms:

The model must give one real-world situation
It must NOT explain theory
It must be concrete and practical

👉 Purpose:
Force the model to stop being abstract and instead say things like:

“A bank uses machine learning to detect fraud in credit card transactions.”

🔧 Part 3 — Fallback Prompt

This is the default behavior when routing fails or is unclear.

In simple terms:

If the system doesn’t know what to do, it still responds
The response must be short and simple
It avoids vague or empty answers

👉 Purpose:
Act as a safety net, so the system never breaks or produces nonsense when routing logic fails.

🧭 Final intuition

This cell is about:

“We are not making the system smarter — we are making its instructions better.”

So instead of changing:

model ❌
architecture ❌

You are improving:

clarity ✔
structure ✔
consistency ✔
usefulness of outputs ✔

In [11]:
# ==========================================================
# CELL 26 — PROMPT IMPROVEMENT (LCEL QUALITY UPGRADE)
# ----------------------------------------------------------
# PURPOSE:
# Improve the quality of responses without changing the
# architecture of the LCEL router system.
#
# WHAT THIS DOES:
# - Makes explanations clearer and more structured
# - Forces concrete examples (not vague answers)
# - Improves consistency of fallback responses
# ==========================================================

from langchain_core.prompts import PromptTemplate

# ----------------------------------------------------------
# 1. EXPLAIN PROMPT (improved teaching behavior)
# ----------------------------------------------------------
# Goal:
# - Simple explanation
# - Structured thinking
# - Avoid vague definitions
explain_prompt = PromptTemplate.from_template(
    """
You are a helpful and clear teacher.

Explain the topic in a simple way for a beginner.

Rules:
- Use 2 to 3 short sentences
- Avoid repetition
- Use a real-world analogy if possible

Topic: {topic}
"""
)

# ----------------------------------------------------------
# 2. EXAMPLE PROMPT (make answers concrete)
# ----------------------------------------------------------
# Goal:
# - Force real-world grounding
# - Avoid abstract or generic answers
example_prompt = PromptTemplate.from_template(
    """
You are a teacher giving practical learning examples.

Give ONE clear real-world example of the topic.

Rules:
- Be specific (use a real situation or scenario)
- Keep it in 2–3 sentences max
- Do not explain theory, only example

Topic: {topic}
"""
)

# ----------------------------------------------------------
# 3. FALLBACK PROMPT (control weak routing cases)
# ----------------------------------------------------------
# Goal:
# - Prevent empty or vague outputs
# - Force short structured response
fallback_prompt = PromptTemplate.from_template(
    """
You are a helpful assistant.

Answer the topic in a simple and direct way.

Rules:
- 1 to 2 short sentences only
- Be clear and concrete
- Avoid vague statements

Topic: {topic}
"""
)

In [12]:
# ==========================================================
# CELL 26 — PROMPT IMPROVEMENT (LCEL QUALITY UPGRADE)
# ----------------------------------------------------------
# PURPOSE:
# Improve the quality of responses without changing the
# architecture of the LCEL router system.
#
# WHAT THIS DOES:
# - Makes explanations clearer and more structured
# - Forces concrete examples (not vague answers)
# - Improves consistency of fallback responses
# ==========================================================

from langchain_core.prompts import PromptTemplate

# ----------------------------------------------------------
# 1. EXPLAIN PROMPT (improved teaching behavior)
# ----------------------------------------------------------
# Goal:
# - Simple explanation
# - Structured thinking
# - Avoid vague definitions
explain_prompt = PromptTemplate.from_template(
    """
You are a helpful and clear teacher.

Explain the topic in a simple way for a beginner.

Rules:
- Use 2 to 3 short sentences
- Avoid repetition
- Use a real-world analogy if possible

Topic: {topic}
"""
)

# ----------------------------------------------------------
# 2. EXAMPLE PROMPT (make answers concrete)
# ----------------------------------------------------------
# Goal:
# - Force real-world grounding
# - Avoid abstract or generic answers
example_prompt = PromptTemplate.from_template(
    """
You are a teacher giving practical learning examples.

Give ONE clear real-world example of the topic.

Rules:
- Be specific (use a real situation or scenario)
- Keep it in 2–3 sentences max
- Do not explain theory, only example

Topic: {topic}
"""
)

# ----------------------------------------------------------
# 3. FALLBACK PROMPT (control weak routing cases)
# ----------------------------------------------------------
# Goal:
# - Prevent empty or vague outputs
# - Force short structured response
fallback_prompt = PromptTemplate.from_template(
    """
You are a helpful assistant.

Answer the topic in a simple and direct way.

Rules:
- 1 to 2 short sentences only
- Be clear and concrete
- Avoid vague statements

Topic: {topic}
"""
)

In [15]:
result = full_chain.invoke({"topic": "machine learning example"})
print(result)

I can only explain topics for now.


--------------------------------------------------------------------------------------------------------
This cell is a critical compatibility fix between LangChain (LCEL) and Hugging Face models.

The problem is simple:

LangChain sends structured data (like dictionaries), but Hugging Face tokenizer only accepts raw strings.

So this cell ensures that everything is converted into a clean string before reaching the model, preventing crashes and inconsistent behavior.

🔧 Part 1 — Purpose & Problem Definition

This section explains the why of the cell.

LangChain LCEL passes data as structured objects (often dictionaries)
Hugging Face models require plain text strings
If you don’t fix this, you get errors or unpredictable behavior

👉 So this cell exists to bridge two incompatible systems:
LangChain → HF Transformers

🛠️ Part 2 — Safe Wrapper (run_llm)

This is the core fix of the entire system.

What it does:

Checks if input is a dictionary
Extracts "topic" safely
Converts everything into a clean string prompt
Sends that string to the tokenizer and model

👉 Conceptually:
This is your data normalization layer

It guarantees that:

No matter what LCEL sends, the model always receives valid text.

🔗 Part 3 — Prompt Construction Inside the Function

Instead of relying only on PromptTemplate, you manually rebuild the prompt:

Explain this topic simply:
{topic}

Why?

Ensures full control over final input
Prevents LangChain structure from leaking into HF model
Guarantees consistency

👉 Conceptually:
This is a manual override of prompt control for stability

⚙️ Part 4 — Tokenization + Model Execution

This is the Hugging Face inference layer:

tokenizer() → converts text into tokens
model.generate() → produces output tokens
decode() → converts tokens back into readable text

👉 Conceptually:
This is the actual “AI brain execution step”

Everything before this is just preparation.

🔗 Part 5 — Rebuilding the LCEL Chain
explain_chain = explain_prompt | RunnableLambda(run_llm) | StrOutputParser()

This rebuilds the pipeline using LCEL:

Prompt → Safe LLM execution → Clean output

But now:

The LLM step is safe
It handles LCEL → HF mismatch correctly
It prevents crashes from structured inputs

👉 Conceptually:
This is your stable LCEL production pipeline

🧪 Part 6 — Test Execution
result = explain_chain.invoke({"topic": "machine learning"})

This tests the full system:

LCEL sends dictionary input
PromptTemplate formats it
run_llm() safely extracts text
Hugging Face generates output
Result is cleaned and printed

👉 Conceptually:
This is the end-to-end validation of your full AI pipeline

🧭 Final intuition

This cell is not about improving intelligence — it is about:

Making LangChain and Hugging Face “speak the same language”

It fixes the most important engineering issue in your notebook:

✔ input compatibility
✔ data normalization
✔ stable LCEL execution
✔ safe model integration

This is what makes your system move from “prototype that breaks” → “stable architecture”

In [16]:
# ==========================================================
# CELL 27 — FIX LCEL → HF TOKENIZER INTERFACE (CRITICAL FIX)
# ----------------------------------------------------------
# PURPOSE:
# Ensure PromptTemplate output is converted into a proper
# string BEFORE being passed to HuggingFace tokenizer.
#
# WHY THIS IS NEEDED:
# LangChain passes structured inputs (dict-like),
# but HF tokenizer requires a raw string.
# ==========================================================

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# ----------------------------------------------------------
# SAFE WRAPPER (THIS IS THE KEY FIX)
# ----------------------------------------------------------
def run_llm(inputs):
    """
    Converts LangChain structured input → clean string → HF model
    """

    # If LCEL sends dict, extract topic safely
    if isinstance(inputs, dict):
        topic = inputs.get("topic", "")
    else:
        topic = str(inputs)

    # Build final prompt string manually (IMPORTANT FIX)
    prompt_text = f"Explain this topic simply:\n{topic}"

    # Tokenize + generate
    inputs_tok = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs_tok, max_new_tokens=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ----------------------------------------------------------
# REBUILD CHAIN (KEEP SAME STRUCTURE)
# ----------------------------------------------------------
explain_chain = explain_prompt | RunnableLambda(run_llm) | StrOutputParser()


# ----------------------------------------------------------
# TEST
# ----------------------------------------------------------
result = explain_chain.invoke({"topic": "machine learning"})
print("=== OUTPUT ===")
print(result)

=== OUTPUT ===
Using a real-world analogy if possible


In [14]:
# ==========================================================
# CELL 28 — CLEAN LCEL CHAIN (NO MANUAL TOKENIZER)
# ----------------------------------------------------------
# PURPOSE:
# Replace manual HuggingFace function with a clean LCEL-compatible
# runnable function.
#
# WHY:
# - Removes ValueError issues from tokenizer
# - Aligns architecture with real LangChain LCEL design
# - Makes pipeline stable and predictable
# ==========================================================

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# ----------------------------------------------------------
# SIMPLE SAFE LLM WRAPPER (LCEL COMPATIBLE)
# ----------------------------------------------------------
def llm_runnable(input_dict):
    """
    Receives LCEL dict input → extracts topic → runs HF model safely
    """

    # Extract topic safely from LCEL input
    topic = input_dict.get("topic", "")

    # Build clean prompt
    prompt_text = f"Explain this topic simply in 2-3 sentences:\n{topic}"

    # HuggingFace inference (SAFE + CONSISTENT)
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ----------------------------------------------------------
# FINAL LCEL EXPLAIN CHAIN
# ----------------------------------------------------------
explain_chain = explain_prompt | RunnableLambda(llm_runnable) | StrOutputParser()

print("LCEL explain_chain ready ✔")

LCEL explain_chain ready ✔


In [15]:
# ==========================================================
# CELL 28 — CLEAN LCEL CHAIN (NO MANUAL TOKENIZER)
# ----------------------------------------------------------
# PURPOSE:
# Replace manual HuggingFace function with a clean LCEL-compatible
# runnable function.
#
# WHY:
# - Removes ValueError issues from tokenizer
# - Aligns architecture with real LangChain LCEL design
# - Makes pipeline stable and predictable
# ==========================================================

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# ----------------------------------------------------------
# SIMPLE SAFE LLM WRAPPER (LCEL COMPATIBLE)
# ----------------------------------------------------------
def llm_runnable(input_dict):
    """
    Receives LCEL dict input → extracts topic → runs HF model safely
    """

    # Extract topic safely from LCEL input
    topic = input_dict.get("topic", "")

    # Build clean prompt
    prompt_text = f"Explain this topic simply in 2-3 sentences:\n{topic}"

    # HuggingFace inference (SAFE + CONSISTENT)
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ----------------------------------------------------------
# FINAL LCEL EXPLAIN CHAIN
# ----------------------------------------------------------
explain_chain = explain_prompt | RunnableLambda(llm_runnable) | StrOutputParser()

print("LCEL explain_chain ready ✔")

LCEL explain_chain ready ✔


In [16]:
# ==========================================================
# CELL 30 — FINAL DEMO (NOTEBOOK COMPLETION)
# ----------------------------------------------------------
# PURPOSE:
# Single clean demonstration of entire system:
# input → router → chain → output
#
# THIS IS THE END OF NOTEBOOK 1
# ==========================================================

inputs = [
    {"topic": "machine learning"},
    {"topic": "machine learning example"},
    {"topic": "explain deep learning"}
]

for inp in inputs:
    print("\nINPUT:", inp["topic"])
    print("OUTPUT:", router_chain.invoke(inp))
    print("-" * 50)


INPUT: machine learning
OUTPUT: fallback
--------------------------------------------------

INPUT: machine learning example
OUTPUT: example
--------------------------------------------------

INPUT: explain deep learning
OUTPUT: explain
--------------------------------------------------
